In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "NOTEBOOKS":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "DATA"
RAW_DIR = DATA_DIR / "RAW"
PROCESSED_DIR = DATA_DIR / "PROCESSED"
MODELS_DIR = PROJECT_ROOT / "MODELS"
OUTPUTS_DIR = PROJECT_ROOT / "OUTPUTS"
PLOTS_DIR = OUTPUTS_DIR / "PLOTS"

for d in [PROCESSED_DIR, MODELS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

xtr_path = PROCESSED_DIR / "X_train_engineered.csv"
xte_path = PROCESSED_DIR / "X_test_engineered.csv"
ytr_path = PROCESSED_DIR / "y_train.csv"
yte_path = PROCESSED_DIR / "y_test.csv"

X_train = pd.read_csv(xtr_path)
X_test = pd.read_csv(xte_path)
y_train = pd.read_csv(ytr_path).squeeze("columns").astype(int)
y_test = pd.read_csv(yte_path).squeeze("columns").astype(int)

Project root: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\PREDICTIVE-MAINTENANCE
Python: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]


In [2]:
# Mutual information requires numeric features.
X_train_num = X_train.select_dtypes(include=np.number).copy()
X_test_num = X_test[X_train_num.columns].copy()

# Fill any remaining missing values using training medians.
X_train_num = X_train_num.fillna(X_train_num.median())
X_test_num = X_test_num.fillna(X_train_num.median())

# ---------------------------------------------------------
# 1. Mutual Information Ranking
# ---------------------------------------------------------
mi = mutual_info_classif(
    X_train_num,
    y_train,
    random_state=42
)

mi_scores = (
    pd.Series(mi, index=X_train_num.columns)
    .sort_values(ascending=False)
)

print("Feature ranking using Mutual Information:")
display(mi_scores.to_frame("mutual_information"))

Feature ranking using Mutual Information:


,mutual_information
torque_[nm],0.044576
rotational_speed_[rpm],0.028642
tool_wear_[min],0.014164
air_temperature_[k],0.006120
type,0.004145
twf,0.003470
pwf,0.003288
process_temperature_[k],0.001930
hdf,0.001803
rnf,0.000740


In [3]:
corr = X_train_num.corr().abs()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

# Features having correlation >= 0.90 with another feature
correlated_features = [
    column
    for column in upper.columns
    if any(upper[column] >= 0.90)
]

print("Highly correlated features:")
print(correlated_features)

# Keep the feature with the better mutual-information score
features_after_correlation = [
    feature for feature in X_train_num.columns
    if feature not in correlated_features
]

print("\nFeatures after correlation filtering:")
print(features_after_correlation)

filtered_mi_scores = (
    mi_scores[
        mi_scores.index.isin(features_after_correlation)
    ]
    .sort_values(ascending=False)
)

Highly correlated features:
['torque_[nm]']

Features after correlation filtering:
['type', 'air_temperature_[k]', 'process_temperature_[k]', 'rotational_speed_[rpm]', 'tool_wear_[min]', 'twf', 'hdf', 'pwf', 'osf', 'rnf']


In [4]:
TOP_N = min(10, len(filtered_mi_scores))

selected_features = filtered_mi_scores.head(TOP_N).index.tolist()

print("\nFinal selected features:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")


X_train_selected = X_train_num[selected_features].copy()
X_test_selected = X_test_num[selected_features].copy()


pd.Series(
    selected_features,
    name="feature"
).to_csv(
    PROCESSED_DIR / "selected_features.csv",
    index=False
)

X_train_selected.to_csv(
    PROCESSED_DIR / "X_train_selected.csv",
    index=False
)

X_test_selected.to_csv(
    PROCESSED_DIR / "X_test_selected.csv",
    index=False
)

print("\nSaved selected feature datasets.")


Final selected features:
1. rotational_speed_[rpm]
2. tool_wear_[min]
3. air_temperature_[k]
4. type
5. twf
6. pwf
7. process_temperature_[k]
8. hdf
9. rnf
10. osf

Saved selected feature datasets.
